# Test — Google Gemini API key

Confirms the `GOOGLE_API_KEY` in `backend/.env` is valid and can make real calls.

**Prereqs**
- Run with the project venv kernel (`backend/.venv`). It needs `ipykernel`: `uv add --dev ipykernel`.
- The key is read from `backend/.env` — it is never hard-coded or printed here.
- Cell 2 (text) makes a tiny **paid** call (Gemini Flash). The image cell is optional and paid.
- Clear outputs before committing this notebook.

In [ ]:
import os
from dotenv import load_dotenv, find_dotenv

# Load backend/.env by searching upward from this notebook's directory.
load_dotenv(find_dotenv(usecwd=True))
api_key = os.environ.get("GOOGLE_API_KEY")
assert api_key, "GOOGLE_API_KEY not found - is backend/.env present and populated?"
print("Key loaded:", bool(api_key), "| length:", len(api_key))

from google import genai

client = genai.Client(api_key=api_key)


In [ ]:
# 1) Connectivity (free): list available models
models = list(client.models.list())
print(f"Reachable - {len(models)} models available. First few:")
for m in models[:8]:
    print(" -", m.name)

In [ ]:
# 2) Generation (cheap paid call): Gemini Flash text
resp = client.models.generate_content(
    model="gemini-2.5-flash",
    contents="Reply with exactly: Blue Fit API key works.",
)
print(resp.text)

## Optional — image generation (Nano Banana)

Paid call that returns an actual image (the product's real use case). Run only if you want to confirm image access on this key.

In [ ]:
# 3) OPTIONAL paid call: Nano Banana image generation
from IPython.display import Image, display

img_resp = client.models.generate_content(
    model="gemini-2.5-flash-image",
    contents="A serene sunrise over calm open water, ocean-blue tones, cinematic wellness aesthetic.",
)
shown = False
for part in img_resp.candidates[0].content.parts:
    data = getattr(getattr(part, "inline_data", None), "data", None)
    if data:
        display(Image(data=data))
        shown = True
    elif getattr(part, "text", None):
        print(part.text)
print("Image returned." if shown else "No image part in response.")